# A Multi-Axis Framework for Evaluating the Faithfulness, Stability, and Fairness of SHAP and LIME Explanations in Mental-Health Treatment Prediction

Implementation of the paper's pipeline, section by section:

| Notebook section | Paper section | Output |
|---|---|---|
| 1. Data | III-A | Fig. 1 |
| 2. Leak-free preprocessing | III-B | `LeakFreePreprocessor` |
| 3. Class imbalance (SMOTENC) | III-C | balanced training set |
| 4. Models + 5-fold CV | III-D, IV-A | Table I |
| 5. Statistical comparison | III-E | Friedman + Holm-corrected Wilcoxon |
| 6. Hold-out performance | IV-A | Table II |
| 7. SHAP and LIME explainers | III-F | attributions |
| 8. Axis 1: faithfulness | III-G(1), IV-B | Table III, Fig. 2 |
| 9. Axis 2: stability | III-G(2), IV-C | Table IV, Fig. 4 |
| 10. Axis 3: SHAP–LIME agreement | III-G(3), IV-D | Table V, Fig. 3 |
| 11. Axis 4: fairness | III-G(4), IV-E, IV-F | Tables VI and VII, Fig. 5 |
| 12. Global importance, interactions, contrastive case | IV-G | Fig. 6 |

**Data.** Download `Mental Health Dataset.csv` from
<https://www.kaggle.com/datasets/bhavikjikadara/mental-health-dataset> and put it next to this notebook
(or set `DATA_PATH` below). On Kaggle, just attach the dataset.

**Choices the paper leaves open** (all are switches in the configuration cell):

- `Days_Indoors` is mapped to an ordinal 0–4 scale and treated as the one non-categorical feature, which is what lets SMOTENC run (it needs at least one continuous column).
- Interventional SHAP uses a random background sample of the pre-resampling training set (`N_BACKGROUND`), and the per-model axes are computed on a random test subsample (`N_EXPLAIN`).
- F1, precision and recall are class-weighted averages.
- AOPC is computed on the predicted-class probability; deletion and insertion curves use hold-out accuracy.

In [ ]:
%pip install -q xgboost lightgbm shap lime imbalanced-learn scikit-learn scipy pandas matplotlib

## 0. Configuration

In [ ]:
import os, glob, time, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             precision_score, recall_score)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, SMOTEN, SMOTENC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")

# ---------------- reproducibility ----------------
SEED = 42
np.random.seed(SEED)

# ---------------- data ----------------
DATA_PATH = None                 # path to the CSV; None = search the usual places
TARGET = "treatment"
PROTECTED = "Gender"
DROP_COLS = ["Timestamp", "comments"]   # non-informative columns (dropped if present)
DROP_DUPLICATES = False          # set True if your copy of the data was de-duplicated
# Ordinal features kept numeric (structural mapping, no dataset statistics involved).
ORDINAL_NUMERIC = {
    "Days_Indoors": ["Go out Every day", "1-14 days", "15-30 days",
                     "31-60 days", "More than 2 months"],
}

# ---------------- evaluation sizes ----------------
TEST_SIZE = 0.20
N_FOLDS = 5
N_BACKGROUND = 100       # SHAP / interaction background sample (pre-resampling train set)
N_EXPLAIN = 2000         # test instances explained per model (faithfulness, stability)
N_FAIR = None            # test instances for the best-model fairness audit (None = all)
N_STEPS = 10             # deletion / insertion perturbation steps
N_PAIRS = 500            # instance pairs for the Lipschitz estimate
N_LIME = 100             # instances for SHAP-LIME agreement
LIME_NUM_SAMPLES = 5000  # LIME neighbourhood size
N_LIME_STAB = 20         # instances for LIME repeat-variance
N_LIME_REPEATS = 10      # repeated LIME runs per instance
N_INTERACT = 100         # instances for SHAP interaction values

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- plotting ----------------
MODEL_COLORS = {"XGBoost": "#2a78d6", "LightGBM": "#eb6834",
                "Random Forest": "#1baf7a", "Logistic Regression": "#eda100"}
GROUP_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.titlesize": 11, "axes.labelsize": 10,
    "legend.frameon": False, "lines.linewidth": 2,
})

def save_table(df, name):
    df.to_csv(os.path.join(OUT_DIR, name + ".csv"))
    return df

def save_fig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, name + ".png"))

## 1. Data (Section III-A)

In [ ]:
def locate_csv(path=None):
    if path:
        return path
    patterns = ["Mental Health Dataset.csv", "*ental*ealth*.csv", "data/*ental*ealth*.csv",
                "/kaggle/input/**/*ental*ealth*.csv", "/content/**/*ental*ealth*.csv"]
    for p in patterns:
        hits = sorted(glob.glob(p, recursive=True))
        if hits:
            return hits[0]
    try:                                    # optional: needs Kaggle credentials
        import kagglehub
        d = kagglehub.dataset_download("bhavikjikadara/mental-health-dataset")
        hits = sorted(glob.glob(os.path.join(d, "**", "*.csv"), recursive=True))
        if hits:
            return hits[0]
    except Exception:
        pass
    raise FileNotFoundError(
        "Dataset not found. Download 'Mental Health Dataset.csv' from "
        "https://www.kaggle.com/datasets/bhavikjikadara/mental-health-dataset "
        "and set DATA_PATH to its location.")

csv_path = locate_csv(DATA_PATH)
raw = pd.read_csv(csv_path)
raw.columns = [c.strip() for c in raw.columns]
print(f"Loaded {csv_path}: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

In [ ]:
def find_col(df, name):
    # Case-insensitive column lookup; returns None when absent.
    for c in df.columns:
        if c.lower() == name.lower():
            return c
    return None

def normalize_gender(value):
    # Free-text gender -> fixed vocabulary, using structural rules only.
    if pd.isna(value):
        return "Other"
    s = str(value).strip().lower()
    if s in {"f", "female", "woman", "cis female", "cis-female", "femail", "female (cis)"}:
        return "Female"
    if s in {"m", "male", "man", "cis male", "cis-male", "mail", "male (cis)", "malr", "maile", "make"}:
        return "Male"
    return "Other"

def clean(df):
    # Structural cleaning only: nothing here is learned from the data.
    df = df.copy()
    for name in DROP_COLS:
        col = find_col(df, name)
        if col is not None:
            df = df.drop(columns=col)
    for c in df.columns:                                   # tidy string cells
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype("object").where(df[c].notna(), np.nan)
            df[c] = df[c].map(lambda v: v.strip() if isinstance(v, str) else v)

    tcol = find_col(df, TARGET)
    y = df[tcol].map(lambda v: {"yes": 1, "no": 0, "1": 1, "0": 0}.get(str(v).strip().lower(), np.nan))
    n_missing = int(y.isna().sum())
    df = df.loc[y.notna()].copy()
    df[tcol] = y[y.notna()].astype(int)
    print(f"Rows removed for a missing target: {n_missing}")

    gcol = find_col(df, PROTECTED)
    if gcol is not None:
        df[gcol] = df[gcol].map(normalize_gender)

    for name, order in ORDINAL_NUMERIC.items():
        col = find_col(df, name)
        if col is not None:
            mapping = {v.lower(): i for i, v in enumerate(order)}
            df[col] = df[col].map(lambda v: mapping.get(str(v).strip().lower(), np.nan)).astype(float)

    if DROP_DUPLICATES:
        before = len(df)
        df = df.drop_duplicates()
        print(f"Duplicate rows removed: {before - len(df):,}")
    return df.reset_index(drop=True)

data = clean(raw)
TARGET_COL = find_col(data, TARGET)
GENDER_COL = find_col(data, PROTECTED)
FEATURES = [c for c in data.columns if c != TARGET_COL]
NUMERIC_COLS = [c for c in FEATURES if pd.api.types.is_numeric_dtype(data[c])]
CAT_COLS = [c for c in FEATURES if c not in NUMERIC_COLS]
CAT_IDX = [FEATURES.index(c) for c in CAT_COLS]

counts = data[TARGET_COL].value_counts()
print(f"N = {len(data):,} | features = {len(FEATURES)} "
      f"({len(CAT_COLS)} categorical, {len(NUMERIC_COLS)} numeric: {NUMERIC_COLS})")
print(f"Positive: {counts.get(1, 0):,} ({counts.get(1, 0) / len(data):.1%}) | "
      f"Negative: {counts.get(0, 0):,} ({counts.get(0, 0) / len(data):.1%})")

In [ ]:
# Fig. 1 - target and gender distributions (plus age when the column exists)
age_col = find_col(data, "Age")
panels = 3 if age_col else 2
fig, axes = plt.subplots(1, panels, figsize=(4.2 * panels, 3.4))

tc = data[TARGET_COL].map({1: "Sought treatment", 0: "No treatment"}).value_counts()
axes[0].bar(tc.index, tc.values, color="#2a78d6", width=0.55)
for i, v in enumerate(tc.values):
    axes[0].text(i, v, f"{v:,}\n({v / len(data):.1%})", ha="center", va="bottom", fontsize=9)
axes[0].set_ylim(0, tc.max() * 1.22); axes[0].set_title("Target class distribution"); axes[0].set_ylabel("Respondents")

gc = data[GENDER_COL].value_counts()
axes[1].bar(gc.index, gc.values, color="#2a78d6", width=0.55)
for i, v in enumerate(gc.values):
    axes[1].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
axes[1].set_ylim(0, gc.max() * 1.15); axes[1].set_title("Self-reported gender")

if age_col:
    axes[2].hist(pd.to_numeric(data[age_col], errors="coerce").dropna().clip(10, 90), bins=30, color="#2a78d6")
    axes[2].set_title("Age distribution"); axes[2].set_xlabel("Age")
fig.tight_layout(); save_fig(fig, "fig1_distributions"); plt.show()

## 2. Leak-free preprocessing (Section III-B)

The stratified 80/20 split happens **before** any encoder or imputer is fit. `LeakFreePreprocessor` then fits one
`LabelEncoder` per categorical column and one median imputer per column on training rows only. Categories that never
appeared in training are treated as missing and imputed.

In [ ]:
class LeakFreePreprocessor(BaseEstimator, TransformerMixin):
    # Per-column LabelEncoder + per-column median imputer, fit on training data only.

    def __init__(self, numeric_cols=None):
        self.numeric_cols = numeric_cols

    def _encode(self, X):
        out = np.full((len(X), len(self.columns_)), np.nan, dtype=float)
        for j, c in enumerate(self.columns_):
            if c in self.encoders_:
                mapping = self.mappings_[c]
                out[:, j] = X[c].map(lambda v: mapping.get(str(v), np.nan) if pd.notna(v) else np.nan).to_numpy(dtype=float)
            else:
                out[:, j] = pd.to_numeric(X[c], errors="coerce").to_numpy(dtype=float)
        return out

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        numeric = set(self.numeric_cols or [])
        self.columns_ = list(X.columns)
        self.encoders_, self.mappings_ = {}, {}
        for c in self.columns_:
            if c not in numeric:
                le = LabelEncoder().fit(X[c].dropna().astype(str))
                self.encoders_[c] = le
                self.mappings_[c] = {cls: i for i, cls in enumerate(le.classes_)}
        enc = self._encode(X)
        med = np.nanmedian(enc, axis=0)
        cat_mask = np.array([c in self.encoders_ for c in self.columns_])
        med[cat_mask] = np.round(med[cat_mask])          # keep imputed codes valid categories
        self.medians_ = np.where(np.isnan(med), 0.0, med)
        self.categorical_idx_ = list(np.where(cat_mask)[0])
        return self

    def transform(self, X):
        enc = self._encode(pd.DataFrame(X)[self.columns_])
        return np.where(np.isnan(enc), self.medians_, enc)

    def get_feature_names_out(self, input_features=None):
        return np.array(self.columns_)


# 80/20 stratified split BEFORE fitting anything
X_all, y_all = data[FEATURES], data[TARGET_COL].to_numpy()
Xtr_df, Xte_df, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, stratify=y_all, random_state=SEED)
Xtr_df, Xte_df = Xtr_df.reset_index(drop=True), Xte_df.reset_index(drop=True)
gender_test = Xte_df[GENDER_COL].to_numpy()

prep = LeakFreePreprocessor(numeric_cols=NUMERIC_COLS).fit(Xtr_df)
X_train = prep.transform(Xtr_df)      # pre-resampling training set (also the explainer background)
X_test = prep.transform(Xte_df)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

## 3. Class-imbalance handling (Section III-C)

SMOTENC is applied only to training data: once on the training split for the final models, and again inside every
cross-validation fold through an imbalanced-learn pipeline, so no synthetic row can cross a fold boundary.

In [ ]:
def make_sampler():
    n = len(FEATURES)
    if 0 < len(CAT_IDX) < n:
        return SMOTENC(categorical_features=CAT_IDX, random_state=SEED)
    if len(CAT_IDX) == n:
        # All-categorical fallback. SMOTEN builds a full pairwise distance matrix, so it needs a lot of memory.
        return SMOTEN(random_state=SEED)
    return SMOTE(random_state=SEED)

t0 = time.time()
X_res, y_res = make_sampler().fit_resample(X_train, y_train)
print(f"{type(make_sampler()).__name__}: {len(X_train):,} -> {len(X_res):,} balanced training instances "
      f"({np.bincount(y_res)}) in {time.time() - t0:.0f}s")

## 4. Predictive models and 5-fold cross-validation (Sections III-D, IV-A)

In [ ]:
def make_models():
    return {
        "XGBoost": XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                                 random_state=SEED, eval_metric="logloss", n_jobs=-1,
                                 enable_categorical=False),   # features are already label-encoded
        "LightGBM": LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                                   random_state=SEED, n_jobs=-1, verbose=-1),
        "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10,
                                                random_state=SEED, n_jobs=-1),
        # The scaler lives inside the estimator, so it is fit on the resampled training fold only
        # and every predict_proba call (including LIME's) is scaling-aware.
        "Logistic Regression": SkPipeline([("scaler", StandardScaler()),
                                           ("lr", LogisticRegression(C=1.0, max_iter=1000, random_state=SEED))]),
    }

MODEL_NAMES = list(make_models().keys())
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
scoring = {"AUC": "roc_auc", "F1": "f1_weighted", "Accuracy": "accuracy"}

cv_scores = {}
for name, model in make_models().items():
    t0 = time.time()
    pipe = ImbPipeline([("prep", LeakFreePreprocessor(numeric_cols=NUMERIC_COLS)),
                        ("smote", make_sampler()),
                        ("clf", model)])
    res = cross_validate(pipe, Xtr_df, y_train, cv=cv, scoring=scoring)
    cv_scores[name] = {m: res[f"test_{m}"] for m in scoring}
    print(f"{name:20s} AUC folds: {np.round(cv_scores[name]['AUC'], 4)}  ({time.time() - t0:.0f}s)")

table1 = pd.DataFrame({m: {n: f"{cv_scores[n][m].mean():.4f} ± {cv_scores[n][m].std(ddof=1):.4f}"
                           for n in MODEL_NAMES} for m in scoring})
print("\nTable I - five-fold stratified CV (mean ± SD)")
save_table(table1, "table1_cv_performance")

## 5. Statistical model comparison (Section III-E)

In [ ]:
def holm(pvals):
    # Holm-Bonferroni step-down adjustment.
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj

fold_auc = np.column_stack([cv_scores[n]["AUC"] for n in MODEL_NAMES])   # folds x models
chi2, p_friedman = stats.friedmanchisquare(*fold_auc.T)
print(f"Friedman test on per-fold AUC: chi2 = {chi2:.2f}, p = {p_friedman:.4f}")

rows = []
for a, b in itertools.combinations(range(len(MODEL_NAMES)), 2):
    w, p = stats.wilcoxon(fold_auc[:, a], fold_auc[:, b])
    rows.append({"Model A": MODEL_NAMES[a], "Model B": MODEL_NAMES[b],
                 "Mean AUC diff (A-B)": (fold_auc[:, a] - fold_auc[:, b]).mean(), "W": w, "p": p})
wilcoxon_df = pd.DataFrame(rows)
wilcoxon_df["p (Holm)"] = holm(wilcoxon_df["p"])
wilcoxon_df["Significant (0.05)"] = wilcoxon_df["p (Holm)"] < 0.05
save_table(wilcoxon_df.round(4), "wilcoxon_holm")

## 6. Final models and hold-out performance (Table II)

In [ ]:
models = {}
for name, model in make_models().items():
    models[name] = model.fit(X_res, y_res)

def predict_proba(name, X):
    # Scaling-aware probability wrapper over the raw (label-encoded) feature space.
    return models[name].predict_proba(np.asarray(X, dtype=float))

rows = {}
for name in MODEL_NAMES:
    proba = predict_proba(name, X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows[name] = {"Accuracy": accuracy_score(y_test, pred),
                  "F1": f1_score(y_test, pred, average="weighted"),
                  "AUC": roc_auc_score(y_test, proba),
                  "Precision": precision_score(y_test, pred, average="weighted"),
                  "Recall": recall_score(y_test, pred, average="weighted")}
table2 = pd.DataFrame(rows).T
BEST = table2["AUC"].idxmax()
print(f"Table II - hold-out test set. Best model by hold-out AUC: {BEST}")
save_table(table2.round(3), "table2_holdout_performance")

## 7. Explanation generation (Section III-F)

- Tree ensembles: `shap.TreeExplainer` with interventional perturbation and a background drawn from the
  **pre-resampling** training set.
- Logistic Regression: `shap.LinearExplainer` on the same background after scaling.
- LIME: tabular explainer fit on the pre-resampling training set, with every call routed through the scaling-aware
  `predict_proba` wrapper.

SHAP values are in each explainer's native units: log-odds for XGBoost, LightGBM and Logistic Regression, probability
for Random Forest.

In [ ]:
rng = np.random.default_rng(SEED)
background = X_train[rng.choice(len(X_train), size=min(N_BACKGROUND, len(X_train)), replace=False)]
explain_idx = rng.choice(len(X_test), size=min(N_EXPLAIN, len(X_test)), replace=False)
X_exp, y_exp = X_test[explain_idx], y_test[explain_idx]

def positive_class(values):
    # Reduce any SHAP output layout to (n_samples, n_features) for the positive class.
    if isinstance(values, list):
        values = values[1]
    values = np.asarray(values)
    if values.ndim == 3:
        values = values[:, :, 1]
    return values

def shap_values(name, X):
    model = models[name]
    if isinstance(model, SkPipeline):                       # Logistic Regression
        scaler, lr = model.named_steps["scaler"], model.named_steps["lr"]
        explainer = shap.LinearExplainer(lr, scaler.transform(background))
        return positive_class(explainer.shap_values(scaler.transform(X)))
    explainer = shap.TreeExplainer(model, data=background, feature_perturbation="interventional")
    return positive_class(explainer.shap_values(X, check_additivity=False))

SHAP = {}
for name in MODEL_NAMES:
    t0 = time.time()
    SHAP[name] = shap_values(name, X_exp)
    print(f"SHAP {name:20s} {SHAP[name].shape}  ({time.time() - t0:.0f}s)")

lime_explainer = LimeTabularExplainer(
    X_train, feature_names=FEATURES, class_names=["No treatment", "Treatment"],
    categorical_features=CAT_IDX, mode="classification", random_state=SEED)

def lime_values(name, x):
    # LIME weights for the positive class as a dense vector in feature order.
    exp = lime_explainer.explain_instance(
        x, lambda Z: predict_proba(name, Z), labels=(1,),
        num_features=len(FEATURES), num_samples=LIME_NUM_SAMPLES)
    out = np.zeros(len(FEATURES))
    for j, w in exp.as_map()[1]:
        out[j] = w
    return out

## 8. Axis 1 — Faithfulness (Sections III-G(1), IV-B)

Features are deleted (replaced by their training-set mean) or inserted in descending order of |SHAP| over ten steps,
recording hold-out accuracy at each step. Reported: deletion AUC, insertion AUC, faithfulness score
(insertion AUC − deletion AUC) and AOPC.

In [ ]:
train_mean = X_train.mean(axis=0)

def faithfulness(name, X, y, sv, n_steps=N_STEPS):
    n, f = X.shape
    rank = np.argsort(np.argsort(-np.abs(sv), axis=1), axis=1)   # 0 = most important feature
    fractions = np.linspace(0, 1, n_steps + 1)
    p0 = predict_proba(name, X)
    cls = p0.argmax(axis=1)
    p0_cls = p0[np.arange(n), cls]
    del_acc, ins_acc, prob_drop = [], [], []
    for frac in fractions:
        top = rank < int(round(frac * f))
        X_del = np.where(top, train_mean, X)        # top-k features removed
        X_ins = np.where(top, X, train_mean)        # only top-k features present
        p_del = predict_proba(name, X_del)
        del_acc.append(accuracy_score(y, p_del.argmax(axis=1)))
        ins_acc.append(accuracy_score(y, predict_proba(name, X_ins).argmax(axis=1)))
        prob_drop.append(np.mean(p0_cls - p_del[np.arange(n), cls]))
    del_auc = np.trapezoid(del_acc, fractions)
    ins_auc = np.trapezoid(ins_acc, fractions)
    return {"Del. AUC": del_auc, "Ins. AUC": ins_auc, "Faithf.": ins_auc - del_auc,
            "AOPC": float(np.mean(prob_drop)),
            "curves": (fractions, np.array(del_acc), np.array(ins_acc))}

FAITH = {name: faithfulness(name, X_exp, y_exp, SHAP[name]) for name in MODEL_NAMES}
table3 = pd.DataFrame({n: {k: v for k, v in FAITH[n].items() if k != "curves"} for n in MODEL_NAMES}).T
print("Table III - faithfulness metrics (lower Del. AUC and higher Ins. AUC are better)")
save_table(table3.round(3), "table3_faithfulness")

In [ ]:
# Fig. 2 - deletion and insertion accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for name in MODEL_NAMES:
    fr, d, i = FAITH[name]["curves"]
    axes[0].plot(fr, d, marker="o", ms=4, color=MODEL_COLORS[name], label=name)
    axes[1].plot(fr, i, marker="o", ms=4, color=MODEL_COLORS[name], label=name)
axes[0].set_title("Deletion (top-ranked SHAP features removed)")
axes[1].set_title("Insertion (top-ranked SHAP features added)")
for ax in axes:
    ax.set_xlabel("Fraction of features perturbed")
axes[0].set_ylabel("Hold-out accuracy"); axes[1].legend(loc="lower right")
fig.tight_layout(); save_fig(fig, "fig2_deletion_insertion"); plt.show()

## 9. Axis 2 — Stability (Sections III-G(2), IV-C)

Empirical Lipschitz coefficient of the SHAP attributions over 500 random pairs of test instances:

$$L_{ij} = \frac{\lVert \phi(x_i) - \phi(x_j) \rVert_2}{\lVert z_i - z_j \rVert_2}$$

where $z$ is the z-score-normalised input (training-set mean and SD). For the best model, the variance of repeated LIME
explanations of the same instance is reported too.

The extra `Mean L (scale-normalised)` column divides each model's attributions by their mean L2 norm first. It is not in
the paper; it is there because raw L compares log-odds attributions (boosted trees, Logistic Regression) with
probability attributions (Random Forest), which differ in scale before any stability difference.

In [ ]:
train_std = X_train.std(axis=0)
train_std[train_std == 0] = 1.0
Z_exp = (X_exp - train_mean) / train_std

def sample_pairs(Z, n_pairs, seed=SEED):
    # Random instance pairs with distinct inputs (identical rows give an undefined ratio).
    r = np.random.default_rng(seed)
    pairs, tries = [], 0
    while len(pairs) < n_pairs and tries < 200 * n_pairs:
        i, j = r.integers(0, len(Z), size=2)
        tries += 1
        if i != j and np.linalg.norm(Z[i] - Z[j]) > 1e-12:
            pairs.append((i, j))
    return np.array(pairs)

PAIRS = sample_pairs(Z_exp, N_PAIRS)
d_input = np.linalg.norm(Z_exp[PAIRS[:, 0]] - Z_exp[PAIRS[:, 1]], axis=1)

LIP, rows = {}, {}
for name in MODEL_NAMES:
    sv = SHAP[name]
    LIP[name] = np.linalg.norm(sv[PAIRS[:, 0]] - sv[PAIRS[:, 1]], axis=1) / d_input
    scale = np.linalg.norm(sv, axis=1).mean()
    rows[f"SHAP - {name}"] = {"Mean L": LIP[name].mean(), "95th pct. L": np.percentile(LIP[name], 95),
                              "Mean L (scale-normalised)": (LIP[name] / scale).mean()}

# LIME repeat-variance for the best model
t0 = time.time()
lime_var = []
for k in range(min(N_LIME_STAB, len(X_exp))):
    reps = np.array([lime_values(BEST, X_exp[k]) for _ in range(N_LIME_REPEATS)])
    lime_var.append(reps.var(axis=0).mean())
lime_var = np.array(lime_var)
rows[f"LIME - {BEST} (var.)"] = {"Mean L": lime_var.mean(), "95th pct. L": np.percentile(lime_var, 95),
                                 "Mean L (scale-normalised)": np.nan}
print(f"LIME repeat-variance computed in {time.time() - t0:.0f}s")

table4 = pd.DataFrame(rows).T
print("Table IV - stability (the LIME row is a variance, not a Lipschitz coefficient)")
save_table(table4, "table4_stability")

In [ ]:
# Fig. 4 - Lipschitz coefficients by model, and the best model's full distribution
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
x, w = np.arange(len(MODEL_NAMES)), 0.36
means = [LIP[n].mean() for n in MODEL_NAMES]
p95 = [np.percentile(LIP[n], 95) for n in MODEL_NAMES]
b1 = axes[0].bar(x - w / 2 - 0.01, means, w, color="#2a78d6", label="Mean L")
b2 = axes[0].bar(x + w / 2 + 0.01, p95, w, color="#86b6ef", label="95th percentile L")
axes[0].bar_label(b1, fmt="%.3f", fontsize=8, padding=2); axes[0].bar_label(b2, fmt="%.3f", fontsize=8, padding=2)
axes[0].set_xticks(x); axes[0].set_xticklabels([n.replace(" ", "\n") for n in MODEL_NAMES])
axes[0].set_ylim(0, max(p95) * 1.3)
axes[0].set_ylabel("Empirical Lipschitz coefficient"); axes[0].set_title("SHAP stability by model")
axes[0].legend(loc="upper right", ncol=2)

axes[1].hist(LIP[BEST], bins=40, color=MODEL_COLORS[BEST])
axes[1].axvline(LIP[BEST].mean(), color="#0b0b0b", lw=1.2, ls="--", label=f"Mean = {LIP[BEST].mean():.3f}")
axes[1].set_xlabel("Pairwise Lipschitz ratio"); axes[1].set_ylabel("Pairs")
axes[1].set_title(f"Distribution for {BEST}"); axes[1].legend()
fig.tight_layout(); save_fig(fig, "fig4_stability"); plt.show()

## 10. Axis 3 — SHAP–LIME agreement (Sections III-G(3), IV-D)

For the best model: per-instance Spearman and Kendall correlation between |SHAP| and |LIME|, top-5 Jaccard overlap,
sign agreement, and the global Spearman correlation between mean |SHAP| and mean |LIME|.

In [ ]:
n_lime = min(N_LIME, len(X_exp))
t0 = time.time()
LIME_BEST = np.array([lime_values(BEST, X_exp[k]) for k in range(n_lime)])
SHAP_BEST = SHAP[BEST][:n_lime]
print(f"LIME explanations for {n_lime} instances in {time.time() - t0:.0f}s")

spearman, kendall, jaccard, sign_agree = [], [], [], []
for s, l in zip(SHAP_BEST, LIME_BEST):
    spearman.append(stats.spearmanr(np.abs(s), np.abs(l))[0])
    kendall.append(stats.kendalltau(np.abs(s), np.abs(l))[0])
    top_s, top_l = set(np.argsort(-np.abs(s))[:5]), set(np.argsort(-np.abs(l))[:5])
    jaccard.append(len(top_s & top_l) / len(top_s | top_l))
    nz = (s != 0) & (l != 0)
    sign_agree.append(np.mean(np.sign(s[nz]) == np.sign(l[nz])) if nz.any() else np.nan)
spearman, kendall, jaccard, sign_agree = map(np.array, (spearman, kendall, jaccard, sign_agree))

g_shap, g_lime = np.abs(SHAP_BEST).mean(axis=0), np.abs(LIME_BEST).mean(axis=0)
rho_g, p_g = stats.spearmanr(g_shap, g_lime)

table5 = pd.DataFrame({"Value": {
    "Mean per-instance Spearman rho": f"{np.nanmean(spearman):.3f} ± {np.nanstd(spearman, ddof=1):.3f}",
    "Mean per-instance Kendall tau": f"{np.nanmean(kendall):.3f}",
    "Mean top-5 Jaccard overlap": f"{np.mean(jaccard):.3f}",
    "Mean sign agreement": f"{np.nanmean(sign_agree):.3f}",
    "Global (aggregate) Spearman rho": f"{rho_g:.3f} (p = {p_g:.4f})"}})
print(f"Table V - SHAP-LIME agreement (best model: {BEST}, N = {n_lime})")
save_table(table5, "table5_shap_lime_agreement")

In [ ]:
# Fig. 3 - per-instance Spearman, top-5 Jaccard, and global importance scatter
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))
axes[0].hist(spearman[~np.isnan(spearman)], bins=20, color="#2a78d6")
axes[0].axvline(np.nanmean(spearman), color="#0b0b0b", lw=1.2, ls="--", label=f"Mean = {np.nanmean(spearman):.3f}")
axes[0].set_xlabel("Per-instance Spearman rho"); axes[0].set_ylabel("Instances")
axes[0].set_title("SHAP-LIME rank correlation"); axes[0].legend()

vals, cnts = np.unique(np.round(jaccard, 3), return_counts=True)
axes[1].bar(vals, cnts, width=0.05, color="#2a78d6")
axes[1].axvline(np.mean(jaccard), color="#0b0b0b", lw=1.2, ls="--", label=f"Mean = {np.mean(jaccard):.3f}")
axes[1].set_xlabel("Top-5 Jaccard overlap"); axes[1].set_title("Top-5 feature overlap"); axes[1].legend()

axes[2].scatter(g_shap, g_lime, s=36, color="#2a78d6", edgecolor="white", linewidth=1)
for j in sorted(set(np.argsort(-g_shap)[:5]) | set(np.argsort(-g_lime)[:5])):
    axes[2].annotate(FEATURES[j], (g_shap[j], g_lime[j]), xytext=(5, 4), textcoords="offset points", fontsize=8)
axes[2].set_xlabel("Mean |SHAP|"); axes[2].set_ylabel("Mean |LIME|")
axes[2].set_title(f"Global importance (Spearman rho = {rho_g:.3f})")
fig.tight_layout(); save_fig(fig, "fig3_shap_lime_agreement"); plt.show()

## 11. Axis 4 — Fairness (Sections III-G(4), IV-E, IV-F)

**Outcome level:** per-gender prevalence, TPR, FPR, accuracy and AUC, with demographic-parity, equal-opportunity and
accuracy gaps. **Explanation level:** the Explanation Demographic Parity Gap (EDPG), the difference in mean |SHAP| per
feature between the two largest gender subgroups, tested with Mann–Whitney U and Bonferroni correction.

In [ ]:
# Outcome-level fairness on the full hold-out set
proba_best = predict_proba(BEST, X_test)[:, 1]
pred_best = (proba_best >= 0.5).astype(int)

rows = {}
for g in pd.unique(gender_test):
    m = gender_test == g
    yt, yp = y_test[m], pred_best[m]
    rows[g] = {"n": int(m.sum()), "Prev.": yt.mean(),
               "TPR": yp[yt == 1].mean() if (yt == 1).any() else np.nan,
               "FPR": yp[yt == 0].mean() if (yt == 0).any() else np.nan,
               "Acc.": (yt == yp).mean(),
               "AUC": roc_auc_score(yt, proba_best[m]) if len(np.unique(yt)) == 2 else np.nan,
               "Selection rate": yp.mean()}
table6 = pd.DataFrame(rows).T.sort_values("n", ascending=False)
table6["n"] = table6["n"].astype(int)

big = table6[table6["n"] >= 30]            # ignore tiny subgroups when computing gaps
gap = lambda col: big[col].max() - big[col].min()
print(f"Table VI - outcome-level fairness by {GENDER_COL} ({BEST})")
print(f"Demographic-parity gap = {gap('Selection rate'):.3f} | "
      f"equal-opportunity gap = {gap('TPR'):.3f} | accuracy gap = {gap('Acc.'):.3f}")
save_table(table6.round(3), "table6_outcome_fairness")

In [ ]:
# Explanation-level fairness (EDPG) for the best model
fair_idx = np.arange(len(X_test)) if N_FAIR is None else \
    np.random.default_rng(SEED).choice(len(X_test), size=min(N_FAIR, len(X_test)), replace=False)
t0 = time.time()
SHAP_FAIR = shap_values(BEST, X_test[fair_idx])
g_fair = gender_test[fair_idx]
print(f"SHAP for {len(fair_idx):,} test instances in {time.time() - t0:.0f}s")

grp_a, grp_b = pd.Series(g_fair).value_counts().index[:2]
abs_a, abs_b = np.abs(SHAP_FAIR[g_fair == grp_a]), np.abs(SHAP_FAIR[g_fair == grp_b])

rows = []
for j, feat in enumerate(FEATURES):
    u, p = stats.mannwhitneyu(abs_a[:, j], abs_b[:, j], alternative="two-sided")
    rows.append({"Feature": feat, f"Mean |SHAP| {grp_a}": abs_a[:, j].mean(),
                 f"Mean |SHAP| {grp_b}": abs_b[:, j].mean(),
                 "EDPG": abs_a[:, j].mean() - abs_b[:, j].mean(), "p (raw)": p})
table7 = pd.DataFrame(rows).set_index("Feature")
table7["|EDPG|"] = table7["EDPG"].abs()
table7["p (Bonferroni)"] = np.minimum(1.0, table7["p (raw)"] * len(FEATURES))
table7["Significant?"] = np.where(table7["p (Bonferroni)"] < 0.05, "Yes", "No")
table7 = table7.sort_values("|EDPG|", ascending=False)
print(f"Table VII - EDPG by {GENDER_COL} ({grp_a} minus {grp_b}); "
      f"{(table7['Significant?'] == 'Yes').sum()} of {len(FEATURES)} features significant after Bonferroni")
save_table(table7, "table7_edpg")
table7.style.format({c: "{:.4f}" for c in table7.columns if table7[c].dtype == float} | {"p (raw)": "{:.2e}", "p (Bonferroni)": "{:.2e}"})

In [ ]:
# Fig. 5 - mean |SHAP| per gender subgroup and the resulting EDPG
order = table7.index[::-1]
ypos, h = np.arange(len(order)), 0.38
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
axes[0].barh(ypos + h / 2 + 0.01, table7.loc[order, f"Mean |SHAP| {grp_a}"], h, color=GROUP_COLORS[0], label=grp_a)
axes[0].barh(ypos - h / 2 - 0.01, table7.loc[order, f"Mean |SHAP| {grp_b}"], h, color=GROUP_COLORS[1], label=grp_b)
axes[0].set_yticks(ypos); axes[0].set_yticklabels(order)
axes[0].set_xlabel("Mean |SHAP|"); axes[0].set_title(f"Attribution magnitude by {GENDER_COL}"); axes[0].legend(loc="lower right")

axes[1].barh(ypos, table7.loc[order, "|EDPG|"], 0.6, color="#2a78d6")
for yy, feat in zip(ypos, order):
    mark = "*" if table7.loc[feat, "Significant?"] == "Yes" else "n.s."
    axes[1].text(table7.loc[feat, "|EDPG|"], yy, f"  {mark}", va="center", fontsize=9, color="#52514e")
axes[1].set_xlabel("|EDPG|"); axes[1].set_title("Explanation demographic parity gap (* = significant after Bonferroni)")
fig.tight_layout(); save_fig(fig, "fig5_edpg"); plt.show()

## 12. Global importance, interactions and a contrastive case (Section IV-G)

In [ ]:
# Fig. 6 - SHAP beeswarm summary for the best model on the hold-out set
X_fair_df = pd.DataFrame(X_test[fair_idx], columns=FEATURES)
shap.summary_plot(SHAP_FAIR, X_fair_df, max_display=len(FEATURES), show=False)
fig = plt.gcf(); fig.suptitle(f"SHAP summary - {BEST}", y=1.01, fontsize=11)
save_fig(fig, "fig6_shap_beeswarm"); plt.show()

global_importance = pd.Series(np.abs(SHAP_FAIR).mean(axis=0), index=FEATURES).sort_values(ascending=False)
save_table(global_importance.to_frame("Mean |SHAP|").round(4), "global_importance")

In [ ]:
# Pairwise SHAP interaction values (tree models only), on a small sample
if isinstance(models[BEST], SkPipeline):
    print("The best model is linear, so it is additive by construction: no interaction values to compute.")
else:
    X_int = X_exp[:min(N_INTERACT, len(X_exp))]
    inter = shap.TreeExplainer(models[BEST]).shap_interaction_values(X_int)
    if isinstance(inter, list):
        inter = inter[1]
    inter = np.asarray(inter)
    if inter.ndim == 4:
        inter = inter[..., 1]
    mean_abs = np.abs(inter).mean(axis=0)
    main = np.diag(mean_abs)
    off = mean_abs - np.diag(main)
    pairs = [{"Feature A": FEATURES[i], "Feature B": FEATURES[j], "Mean |interaction|": 2 * off[i, j],
              "Larger main effect": max(main[i], main[j])}
             for i in range(len(FEATURES)) for j in range(i + 1, len(FEATURES))]
    inter_df = pd.DataFrame(pairs).sort_values("Mean |interaction|", ascending=False).head(10).reset_index(drop=True)
    inter_df["Ratio to main effect"] = inter_df["Mean |interaction|"] / inter_df["Larger main effect"]
    print(f"Share of total attribution carried by interactions: {off.sum() / mean_abs.sum():.1%}")
    display(save_table(inter_df.round(4), "shap_interactions_top10"))

In [ ]:
# Contrastive case: the most confident positive vs. the most confident negative prediction
p_exp = predict_proba(BEST, X_exp)[:, 1]
i_pos, i_neg = int(p_exp.argmax()), int(p_exp.argmin())
sv_pos, sv_neg = SHAP[BEST][i_pos], SHAP[BEST][i_neg]

contrast = pd.DataFrame({
    "Value (positive case)": Xte_df.iloc[explain_idx[i_pos]][FEATURES].to_numpy(),
    "Value (negative case)": Xte_df.iloc[explain_idx[i_neg]][FEATURES].to_numpy(),
    "SHAP (positive case)": sv_pos, "SHAP (negative case)": sv_neg,
    "Delta SHAP": np.abs(sv_pos - sv_neg),
    "Opposite sign": np.sign(sv_pos) * np.sign(sv_neg) < 0}, index=FEATURES)
contrast = contrast.sort_values(["Opposite sign", "Delta SHAP"], ascending=False)
print(f"P(treatment): positive case = {p_exp[i_pos]:.3f}, negative case = {p_exp[i_neg]:.3f}")
save_table(contrast, "contrastive_case").head(8)

## Summary of the four-axis audit

In [ ]:
summary = pd.concat([
    table2[["AUC", "Accuracy"]].add_prefix("Hold-out "),
    table3[["Faithf.", "AOPC"]],
    pd.DataFrame({n: {"Mean L": LIP[n].mean(), "95th pct. L": np.percentile(LIP[n], 95)} for n in MODEL_NAMES}).T,
], axis=1)
print(f"Best model by hold-out AUC: {BEST}")
print(f"Most faithful SHAP explanations: {summary['Faithf.'].idxmax()} | most stable: {summary['Mean L'].idxmin()}")
save_table(summary.round(3), "summary_all_axes")